# Nifty 100 Stock Data Analysis and ML Model

This notebook downloads Nifty 100 stock data and prepares it for machine learning models.

## 1. Import Required Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Define Nifty 100 Stock Symbols

In [ ]:
# Nifty 100 stock symbols (NSE - National Stock Exchange of India)
# Note: For yfinance, we need to add '.NS' suffix for NSE stocks

nifty_100_stocks = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'HINDUNILVR.NS',
    'ICICIBANK.NS', 'KOTAKBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'BAJFINANCE.NS',
    'ITC.NS', 'ASIANPAINT.NS', 'AXISBANK.NS', 'LT.NS', 'DMART.NS',
    'SUNPHARMA.NS', 'TITAN.NS', 'ULTRACEMCO.NS', 'NESTLEIND.NS', 'WIPRO.NS',
    'MARUTI.NS', 'HCLTECH.NS', 'BAJAJFINSV.NS', 'ADANIPORTS.NS', 'ONGC.NS',
    'NTPC.NS', 'POWERGRID.NS', 'M&M.NS', 'TATASTEEL.NS', 'TECHM.NS',
    'TATAMOTORS.NS', 'JSWSTEEL.NS', 'INDUSINDBK.NS', 'COALINDIA.NS', 'DIVISLAB.NS',
    'DRREDDY.NS', 'BAJAJ-AUTO.NS', 'CIPLA.NS', 'EICHERMOT.NS', 'GRASIM.NS',
    'BPCL.NS', 'BRITANNIA.NS', 'SHREECEM.NS', 'HINDALCO.NS', 'APOLLOHOSP.NS',
    'UPL.NS', 'HEROMOTOCO.NS', 'ADANIGREEN.NS', 'ADANIENT.NS', 'SBILIFE.NS',
    'HDFCLIFE.NS', 'TATACONSUM.NS', 'PIDILITIND.NS', 'GODREJCP.NS', 'DABUR.NS',
    'AMBUJACEM.NS', 'SIEMENS.NS', 'GAIL.NS', 'IOC.NS', 'DLF.NS',
    'VEDL.NS', 'HAVELLS.NS', 'INDIGO.NS', 'ICICIPRULI.NS', 'BANKBARODA.NS',
    'BAJAJHLDNG.NS', 'BOSCHLTD.NS', 'BERGEPAINT.NS', 'MARICO.NS', 'MCDOWELL-N.NS',
    'SRF.NS', 'COLPAL.NS', 'LUPIN.NS', 'PNB.NS', 'BIOCON.NS',
    'TORNTPHARM.NS', 'PGHH.NS', 'ACC.NS', 'ADANITRANS.NS', 'TATAPOWER.NS',
    'BANDHANBNK.NS', 'OFSS.NS', 'MUTHOOTFIN.NS', 'MOTHERSON.NS', 'HDFCAMC.NS',
    'ABBOTINDIA.NS', 'PETRONET.NS', 'JINDALSTEL.NS', 'ALKEM.NS', 'LICI.NS',
    'CONCOR.NS', 'LAURUSLABS.NS', 'ICICIGI.NS', 'BHEL.NS', 'NMDC.NS',
    'ASTRAL.NS', 'IRCTC.NS', 'ADANIPOWER.NS', 'ZYDUSLIFE.NS', 'INDUSTOWER.NS'
]

print(f"Total stocks to download: {len(nifty_100_stocks)}")
print(f"First 10 stocks: {nifty_100_stocks[:10]}")

Total stocks to download: 100
First 10 stocks: ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'HINDUNILVR.NS', 'ICICIBANK.NS', 'KOTAKBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'BAJFINANCE.NS']


## 3. Download Stock Data (5 Years, Daily Interval)

In [ ]:
# Define the time period - Download from 2020 to present
start_date = datetime(2020, 1, 1)
end_date = datetime.now()

print(f"Downloading data from {start_date.date()} to {end_date.date()}")
print("This may take a few minutes...\n")

# Dictionary to store stock data
stock_data = {}
failed_downloads = []

# Download data for each stock
for i, symbol in enumerate(nifty_100_stocks, 1):
    try:
        print(f"[{i}/{len(nifty_100_stocks)}] Downloading {symbol}...", end=" ")
        data = yf.download(symbol, start=start_date, end=end_date, interval='1d', progress=False)
        
        if not data.empty:
            # Flatten multi-level columns (yfinance returns MultiIndex columns)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            
            stock_data[symbol] = data
            print(f"✓ ({len(data)} records)")
        else:
            print("✗ (No data)")
            failed_downloads.append(symbol)
    except Exception as e:
        print(f"✗ (Error: {str(e)})")
        failed_downloads.append(symbol)

print(f"\n{'='*60}")
print(f"Successfully downloaded: {len(stock_data)} stocks")
print(f"Failed downloads: {len(failed_downloads)} stocks")
if failed_downloads:
    print(f"Failed stocks: {failed_downloads}")

## 4. Data Exploration and Preprocessing

In [ ]:
# Display sample data for the first stock
if stock_data:
    first_stock = list(stock_data.keys())[0]
    print(f"Sample data for {first_stock}:")
    print(stock_data[first_stock].head())
    print(f"\nData shape: {stock_data[first_stock].shape}")
    print(f"\nColumns: {stock_data[first_stock].columns.tolist()}")
    print(f"\nData types:")
    print(stock_data[first_stock].dtypes)

In [ ]:
# Create a combined dataset with all stocks
combined_data = pd.DataFrame()

for symbol, data in stock_data.items():
    temp_df = data.copy()
    temp_df['Symbol'] = symbol
    temp_df.reset_index(inplace=True)
    combined_data = pd.concat([combined_data, temp_df], ignore_index=True)

print(f"Combined dataset shape: {combined_data.shape}")
print(f"\nColumns: {combined_data.columns.tolist()}")
print(f"\nFirst few rows:")
combined_data.head()

## 5. Feature Engineering for ML Model

In [ ]:
# Create technical indicators and features

def create_features(df):
    """Create features for ML model - predicting RETURNS not prices"""
    df = df.copy()
    
    # Ensure columns are simple strings (not tuples from MultiIndex)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Price-based features (normalized)
    df['Price_Range_Pct'] = (df['High'] - df['Low']) / df['Close'] * 100
    df['Price_Change_Pct'] = ((df['Close'] - df['Open']) / df['Open']) * 100
    df['Gap_Pct'] = ((df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)) * 100
    
    # Returns (lagged)
    df['Return_1d'] = df['Close'].pct_change() * 100
    df['Return_5d'] = df['Close'].pct_change(5) * 100
    df['Return_10d'] = df['Close'].pct_change(10) * 100
    df['Return_20d'] = df['Close'].pct_change(20) * 100
    
    # Moving average ratios (relative to current price)
    df['MA_5'] = df['Close'].rolling(window=5).mean()
    df['MA_10'] = df['Close'].rolling(window=10).mean()
    df['MA_20'] = df['Close'].rolling(window=20).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    
    df['MA5_Ratio'] = (df['Close'] / df['MA_5'] - 1) * 100
    df['MA10_Ratio'] = (df['Close'] / df['MA_10'] - 1) * 100
    df['MA20_Ratio'] = (df['Close'] / df['MA_20'] - 1) * 100
    df['MA50_Ratio'] = (df['Close'] / df['MA_50'] - 1) * 100
    
    # Volatility (as percentage)
    df['Volatility_10'] = df['Return_1d'].rolling(window=10).std()
    df['Volatility_20'] = df['Return_1d'].rolling(window=20).std()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Volume features
    df['Volume_MA10_Ratio'] = df['Volume'] / df['Volume'].rolling(window=10).mean()
    
    # TARGET: Next day's RETURN (not price!)
    df['Target_Return'] = df['Close'].shift(-1) / df['Close'] - 1
    df['Target_Return_Pct'] = df['Target_Return'] * 100
    
    # Target direction (1 if price goes up, 0 if down)
    df['Target_Direction'] = (df['Target_Return'] > 0).astype(int)
    
    return df

# Apply feature engineering to each stock
stock_data_with_features = {}

for symbol, data in stock_data.items():
    stock_data_with_features[symbol] = create_features(data)

# Display sample features
if stock_data_with_features:
    first_stock = list(stock_data_with_features.keys())[0]
    print(f"Features for {first_stock}:")
    print(f"Columns: {stock_data_with_features[first_stock].columns.tolist()}")
    print(f"\nShape: {stock_data_with_features[first_stock].shape}")
    print(f"\nLast 5 rows:")
    print(stock_data_with_features[first_stock].tail())

## 6. Prepare Data for ML Model

In [ ]:
# Combine all stocks with features
ml_data = pd.DataFrame()

for symbol, data in stock_data_with_features.items():
    temp_df = data.copy()
    temp_df['Symbol'] = symbol
    temp_df.reset_index(inplace=True)
    ml_data = pd.concat([ml_data, temp_df], ignore_index=True)

print(f"ML dataset shape (before cleaning): {ml_data.shape}")
print(f"Columns: {ml_data.columns.tolist()}")

# Check for NaN values
print(f"\nNaN count per column:")
print(ml_data.isna().sum())

# Remove rows with NaN values (due to rolling calculations)
ml_data_clean = ml_data.dropna()

print(f"\nML dataset shape (after cleaning): {ml_data_clean.shape}")

# Convert Date column to datetime if not already
ml_data_clean['Date'] = pd.to_datetime(ml_data_clean['Date'])

# Split data: Hold out Dec 2025 and Jan 2026 for future testing
future_test_start = datetime(2025, 12, 1)
print(f"\n{'='*60}")
print("SPLITTING DATA FOR PROPER VALIDATION:")
print(f"Future test period: Dec 2025 - Jan 2026 (held out completely)")

# Training/validation data: everything before Dec 2025
train_val_data = ml_data_clean[ml_data_clean['Date'] < future_test_start].copy()

# Future test data: Dec 2025 and Jan 2026 only
future_test_data = ml_data_clean[ml_data_clean['Date'] >= future_test_start].copy()

print(f"\nTraining/Validation data: {len(train_val_data)} samples (2020 - Nov 2025)")
print(f"Future test data: {len(future_test_data)} samples (Dec 2025 - Jan 2026)")
print(f"Date range in training data: {train_val_data['Date'].min()} to {train_val_data['Date'].max()}")
print(f"Date range in future test: {future_test_data['Date'].min()} to {future_test_data['Date'].max()}")

In [ ]:
# Define features and target - NO PRICE LEAKAGE
feature_columns = [
    'Price_Range_Pct', 'Price_Change_Pct', 'Gap_Pct',
    'Return_1d', 'Return_5d', 'Return_10d', 'Return_20d',
    'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'MA50_Ratio',
    'Volatility_10', 'Volatility_20', 'RSI', 'Volume_MA10_Ratio'
]

# Use only training/validation data (NOT future test data)
X = train_val_data[feature_columns]
y_regression = train_val_data['Target_Return_Pct']  # Predicting % return, NOT price!
y_classification = train_val_data['Target_Direction']  # For direction prediction

# Prepare future test data features and targets
X_future = future_test_data[feature_columns]
y_future_regression = future_test_data['Target_Return_Pct']
y_future_classification = future_test_data['Target_Direction']

print("TARGET: Next day's RETURN (%) - not absolute price!")
print(f"\nTraining/Validation Features shape: {X.shape}")
print(f"Training/Validation Target (return %) shape: {y_regression.shape}")
print(f"\nFuture Test Features shape: {X_future.shape}")
print(f"Future Test Target (return %) shape: {y_future_regression.shape}")
print(f"\nFeatures used (no price leakage):")
for f in feature_columns:
    print(f"  - {f}")

## 7. Split Data into Training and Testing Sets

In [ ]:
# Split training/validation data (80% train, 20% validation)
# Using shuffle=False to maintain time order for time series data
X_train, X_val, y_train_reg, y_val_reg = train_test_split(
    X, y_regression, test_size=0.2, random_state=42, shuffle=True
)

X_train_cls, X_val_cls, y_train_cls, y_val_cls = train_test_split(
    X, y_classification, test_size=0.2, random_state=42, shuffle=True
)

print("DATA SPLIT SUMMARY:")
print("="*60)
print(f"Training set: {X_train.shape[0]} samples (80% of 2020-Nov2025)")
print(f"Validation set: {X_val.shape[0]} samples (20% of 2020-Nov2025)")
print(f"Future test set: {X_future.shape[0]} samples (Dec 2025 - Jan 2026)")
print("="*60)
print("\nThis split ensures:")
print("- Model is trained on 2020-Nov 2025 data only")
print("- Validation during training uses 20% of training period")
print("- Future test (Dec 2025 - Jan 2026) is NEVER seen during training")

## 8. Feature Scaling

In [ ]:
# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_future_scaled = scaler.transform(X_future)

print("Features scaled successfully!")
print(f"Scaled training data shape: {X_train_scaled.shape}")
print(f"Scaled validation data shape: {X_val_scaled.shape}")
print(f"Scaled future test data shape: {X_future_scaled.shape}")

## 9. Train ML Models

In [ ]:
# Train Random Forest Regressor for RETURN prediction
print("Training Random Forest Regressor to predict RETURNS...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train_reg)
print("Random Forest model trained successfully!")

# Make predictions on validation set
y_pred_rf_val = rf_model.predict(X_val_scaled)

# Evaluate on validation set
mse_val = mean_squared_error(y_val_reg, y_pred_rf_val)
mae_val = mean_absolute_error(y_val_reg, y_pred_rf_val)
r2_val = r2_score(y_val_reg, y_pred_rf_val)

print(f"\nRandom Forest - VALIDATION SET Performance:")
print(f"Mean Squared Error: {mse_val:.4f}")
print(f"Mean Absolute Error: {mae_val:.4f}%")
print(f"R² Score: {r2_val:.4f}")

# Direction accuracy (did we predict up/down correctly?)
direction_correct_val = ((y_pred_rf_val > 0) == (y_val_reg > 0)).mean()
print(f"Direction Accuracy: {direction_correct_val:.2%}")

In [ ]:
# Train Gradient Boosting Regressor
print("Training Gradient Boosting Regressor to predict RETURNS...")
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_train_reg)
print("Gradient Boosting model trained successfully!")

# Make predictions on validation set
y_pred_gb_val = gb_model.predict(X_val_scaled)

# Evaluate on validation set
mse_gb_val = mean_squared_error(y_val_reg, y_pred_gb_val)
mae_gb_val = mean_absolute_error(y_val_reg, y_pred_gb_val)
r2_gb_val = r2_score(y_val_reg, y_pred_gb_val)

print(f"\nGradient Boosting - VALIDATION SET Performance:")
print(f"Mean Squared Error: {mse_gb_val:.4f}")
print(f"Mean Absolute Error: {mae_gb_val:.4f}%")
print(f"R² Score: {r2_gb_val:.4f}")

# Direction accuracy
direction_correct_gb_val = ((y_pred_gb_val > 0) == (y_val_reg > 0)).mean()
print(f"Direction Accuracy: {direction_correct_gb_val:.2%}")

## 9.1 Test on Future Data (Dec 2025 - Jan 2026)

This is the TRUE test - evaluating on data the model has NEVER seen during training.

In [ ]:
# TEST ON FUTURE DATA (Dec 2025 - Jan 2026)
print("="*70)
print("FUTURE DATA TEST - Dec 2025 to Jan 2026 (UNSEEN DATA)")
print("="*70)

# Remove any NaN values from future test targets
future_valid_mask = ~y_future_regression.isna()
X_future_valid = X_future_scaled[future_valid_mask]
y_future_valid = y_future_regression[future_valid_mask]

print(f"\nValid future test samples: {len(y_future_valid)}")

if len(y_future_valid) > 0:
    # Random Forest predictions on future data
    y_pred_rf_future = rf_model.predict(X_future_valid)
    
    mse_rf_future = mean_squared_error(y_future_valid, y_pred_rf_future)
    mae_rf_future = mean_absolute_error(y_future_valid, y_pred_rf_future)
    r2_rf_future = r2_score(y_future_valid, y_pred_rf_future)
    dir_acc_rf_future = ((y_pred_rf_future > 0) == (y_future_valid > 0)).mean()
    
    print(f"\nRandom Forest - FUTURE TEST Performance:")
    print(f"Mean Squared Error: {mse_rf_future:.4f}")
    print(f"Mean Absolute Error: {mae_rf_future:.4f}%")
    print(f"R² Score: {r2_rf_future:.4f}")
    print(f"Direction Accuracy: {dir_acc_rf_future:.2%}")
    
    # Gradient Boosting predictions on future data
    y_pred_gb_future = gb_model.predict(X_future_valid)
    
    mse_gb_future = mean_squared_error(y_future_valid, y_pred_gb_future)
    mae_gb_future = mean_absolute_error(y_future_valid, y_pred_gb_future)
    r2_gb_future = r2_score(y_future_valid, y_pred_gb_future)
    dir_acc_gb_future = ((y_pred_gb_future > 0) == (y_future_valid > 0)).mean()
    
    print(f"\nGradient Boosting - FUTURE TEST Performance:")
    print(f"Mean Squared Error: {mse_gb_future:.4f}")
    print(f"Mean Absolute Error: {mae_gb_future:.4f}%")
    print(f"R² Score: {r2_gb_future:.4f}")
    print(f"Direction Accuracy: {dir_acc_gb_future:.2%}")
    
    # Comparison summary
    print("\n" + "="*70)
    print("PERFORMANCE COMPARISON (Predicting Next Day Returns %):")
    print("="*70)
    print(f"{'Metric':<20} {'RF Val':<12} {'RF Future':<12} {'GB Val':<12} {'GB Future':<12}")
    print("-"*70)
    print(f"{'R² Score':<20} {r2_val:<12.4f} {r2_rf_future:<12.4f} {r2_gb_val:<12.4f} {r2_gb_future:<12.4f}")
    print(f"{'MAE (%)':<20} {mae_val:<12.4f} {mae_rf_future:<12.4f} {mae_gb_val:<12.4f} {mae_gb_future:<12.4f}")
    print(f"{'Direction Acc':<20} {direction_correct_val:<12.2%} {dir_acc_rf_future:<12.2%} {direction_correct_gb_val:<12.2%} {dir_acc_gb_future:<12.2%}")
    
    print("\n" + "="*70)
    print("INTERPRETATION:")
    print("="*70)
    print("- R² near 0 is EXPECTED for return prediction (returns are nearly random)")
    print("- Direction Accuracy > 50% suggests some predictive power")
    print("- If Future performance << Validation, model is overfitted")
else:
    print("No valid future test data available.")

## 10. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance - Random Forest Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Visualization - Actual vs Predicted Prices

In [ ]:
# Plot actual vs predicted RETURNS
sample_size = min(100, len(y_val_reg))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_val_reg[:sample_size].values, y_pred_rf_val[:sample_size], alpha=0.5)
plt.plot([-5, 5], [-5, 5], 'r--', label='Perfect prediction')
plt.xlabel('Actual Return (%)')
plt.ylabel('Predicted Return (%)')
plt.title('Validation: Actual vs Predicted Returns (RF)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-5, 5)
plt.ylim(-5, 5)

# Plot for future test data if available
if len(y_future_valid) > 0:
    plt.subplot(1, 2, 2)
    future_sample_size = min(100, len(y_future_valid))
    plt.scatter(y_future_valid[:future_sample_size].values, y_pred_rf_future[:future_sample_size], alpha=0.5)
    plt.plot([-5, 5], [-5, 5], 'r--', label='Perfect prediction')
    plt.xlabel('Actual Return (%)')
    plt.ylabel('Predicted Return (%)')
    plt.title('Future Test (Dec2025-Jan2026): Actual vs Predicted')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)

plt.tight_layout()
plt.show()

## 12. Save Models and Data

In [ ]:
# import pickle

# # Save the trained models
# with open('rf_stock_model.pkl', 'wb') as f:
#     pickle.dump(rf_model, f)

# with open('gb_stock_model.pkl', 'wb') as f:
#     pickle.dump(gb_model, f)

# with open('scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)

# # Save the cleaned data
# ml_data_clean.to_csv('nifty100_stock_data_with_features.csv', index=False)

# print("Models and data saved successfully!")
# print("- rf_stock_model.pkl")
# print("- gb_stock_model.pkl")
# print("- scaler.pkl")
# print("- nifty100_stock_data_with_features.csv")

## Summary

This notebook implements a proper time-series validation approach:

### Data Split Strategy:
1. **Training Data (2020 - Nov 2025)**: Used for model training and validation
   - 80% for training
   - 20% for validation (random split)
2. **Future Test Data (Dec 2025 - Jan 2026)**: Completely held out - NEVER seen during training

### Why This Matters:
- **Prevents Data Leakage**: Future data is not used in training
- **Realistic Evaluation**: Tests model on truly unseen future data
- **Detects Overfitting**: If validation R² is high but future test R² is low, model is overfitted

### Steps Performed:
1. Downloaded 5+ years of daily stock data for Nifty 100 stocks (2020-2026)
2. Created technical indicators (Moving Averages, RSI, Volatility)
3. Split data chronologically to prevent leakage
4. Trained Random Forest and Gradient Boosting models
5. Evaluated on both validation set AND held-out future data
6. Compared performance to detect overfitting